# 05 Diagnose bad jobs: knowledge graph + LLM

**Goal:** for each bad job, ask **"why did this job go wrong, and what should we do?"** and get back a cause, the evidence, and a fix.

This is **retrieval-augmented generation (RAG)** with a knowledge graph:
1. **Retrieve:** walk the graph from the job to collect evidence for each possible cause (alarms before the job, time since the last repair, nozzle hours, feed rate), and compare each one with the guide's rule.
2. **Generate:** give that evidence to the local LLM (Ollama) and ask it to pick the cause and explain, citing the evidence ids.

To see what the graph adds, we run **three versions** on the same jobs:

| mode | what it gets | why |
|---|---|---|
| `kg_rules` | graph evidence, then a simple points score (no LLM) | shows how far the graph alone gets |
| `kg_llm` | graph evidence + guide, answered by the LLM | **your system** |
| `llm_only` | only the job record + guide text, answered by the LLM (no graph, no logs) | shows what the LLM does **without** the graph |

**Before you run:** start Ollama and download a model once, in the Anaconda Prompt or PowerShell:
```
ollama pull llama3.1:8b
```
If Ollama isn't running, the notebook still runs `kg_rules` and skips the two LLM modes.

## 0. Setup

In [1]:
from pathlib import Path
import json, pickle, time
import pandas as pd
import requests

OUT = Path("..") / "data" / "processed"
with open(OUT / "knowledge_graph.pkl", "rb") as f:
    G = pickle.load(f)
key = pd.read_csv(OUT / "answer_key_jobs.csv")

OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.1:8b"          # change to any model you have (run `ollama list` to see them)

try:
    have = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]]
    OLLAMA_OK = any(h.startswith(MODEL) for h in have)
    print("Ollama is running. Models:", have)
    if not OLLAMA_OK:
        print(f"Model {MODEL} not found. Run: ollama pull {MODEL}")
except requests.exceptions.RequestException:
    OLLAMA_OK = False
    print("Ollama is not running, so only kg_rules will run. Start the Ollama app and re-run this cell.")
print(G.number_of_nodes(), "graph nodes |", len(key), "bad jobs in the answer key")

Ollama is running. Models: ['llama3.1:8b', 'qwen3-vl:4b']
817 graph nodes | 99 bad jobs in the answer key


## 1. What is "normal"? (alarm rates)
A 2105 alarm on its own means little, because it happens about 3 times a day. So for each code we compute the **normal count per 24 h** from the whole graph.
Evidence only counts as *unusual* when the 24 h count is clearly above it. This is the same idea as "1.5× normal" in notebook 02.

In [2]:
alarm_times = pd.DataFrame([(d["name"], d["time"]) for _, d in G.nodes(data=True) if d["kind"] == "AlarmEvent"], columns=["code", "time"])
n_days = (alarm_times["time"].max() - alarm_times["time"].min()).days + 1
NORMAL_PER_DAY = (alarm_times["code"].value_counts() / n_days).round(2).to_dict()
NORMAL_PER_DAY

{'2105': 3.89,
 '4001': 1.19,
 '6002': 0.39,
 '1201': 0.33,
 '3302': 0.19,
 '3301': 0.03}

## 2. Retrieve: walk the graph from one job
`retrieve(job_id)` does the graph walk and returns:
- the job's facts and its defect,
- the **candidate causes** for that defect (from the guide's `CAUSES` edges, most likely first),
- for each candidate, **evidence lines** with an id (`E1`, `E2`...), each compared with the guide's rule,
- a simple **points score** per candidate (used by `kg_rules`).

Read the code once. Each block is one walk in the graph.

In [3]:
def out_edges(node, etype):
    return [(t, e) for _, t, e in G.out_edges(node, data=True) if e["type"] == etype]

def retrieve(job_id):
    jn = f"job:{job_id}"
    job = G.nodes[jn]
    defect = [G.nodes[t]["name"] for t, _ in out_edges(jn, "HAS_DEFECT")][0]

    # candidate causes: cause -CAUSES-> defect, filtered by gas where the guide says so
    cands = []
    for c, _, e in G.in_edges(f"defect:{defect}", data=True):
        if e["type"] == "CAUSES" and e["gas"] in ("any", job["gas"]):
            cands.append((e["rank"], G.nodes[c]["name"]))
    if not cands:                                   # e.g. dross on O2 with no O2 row: use all dross causes
        cands = [(e["rank"], G.nodes[c]["name"]) for c, _, e in G.in_edges(f"defect:{defect}", data=True) if e["type"] == "CAUSES"]
    cands = [c for _, c in sorted(set(cands))]

    last_alarm = {e["code"]: (t, e) for t, e in out_edges(jn, "LAST_ALARM")}
    last_repair = {e["cause"]: (t, e) for t, e in out_edges(jn, "AFTER_REPAIR")}
    evidence, scores, n = {}, {}, 0

    def add(cause, text, points):
        nonlocal n
        n += 1
        evidence.setdefault(cause, []).append(f"E{n}: {text}")
        scores[cause] = scores.get(cause, 0) + points

    for cause in cands:
        scores[cause] = 0
        # (a) alarms that the guide says INDICATE this cause, in the 24 h before or during the job
        codes = [G.nodes[s]["name"] for s, _, e in G.in_edges(f"cause:{cause}", data=True) if e["type"] == "INDICATES"]
        for code_ in codes:
            if code_ in last_alarm and last_alarm[code_][1]["hours_before"] <= 24:
                a_node, e = last_alarm[code_]
                normal = NORMAL_PER_DAY.get(code_, 0)
                unusual = e["count_24h"] > max(1.5 * normal, 0.5)
                add(cause, f"alarm {code_} ({G.nodes[f'code:{code_}']['text']}) {e['count_24h']}x in the 24 h before the job, "
                           f"last one {e['hours_before']} h before start; normal is {normal}/day -> {'UNUSUAL' if unusual else 'normal'} [{a_node}]",
                    3 if unusual else 0)
        # (b) collision since the last calibration (alarm 3301 stays relevant until a recalibration RESETS it)
        if cause == "bevel_out_of_calibration" and "3301" in last_alarm:
            a_node, e = last_alarm["3301"]
            rep = last_repair.get(cause)
            not_fixed = rep is None or rep[1]["hours_since"] > e["hours_before"]
            add(cause, f"collision alarm 3301 {e['hours_before']} h before the job; "
                       f"{'NO recalibration since then' if not_fixed else 'recalibrated after it'}; bevel angle of this job {job['bevel_angle']} deg [{a_node}]",
                4 if not_fixed and job["bevel_angle"] != 0 else 0)
        # (c) gas repair after the gas alarms = problem already fixed
        if cause == "gas_supply_low" and cause in last_repair and "1201" in last_alarm:
            rep_node, r = last_repair[cause]
            if r["hours_since"] < last_alarm["1201"][1]["hours_before"]:
                add(cause, f"gas regulator replaced {r['hours_since']} h before the job, after the last 1201 alarm -> supply already fixed [{rep_node}]", -3)
        # (d) time since the last repair, against the guide's interval
        if cause == "glass_dirty":
            if cause in last_repair:
                rep_node, r = last_repair[cause]
                overdue = r["hours_since"] > 24
                add(cause, f"last glass cleaning {r['hours_since']} h before the job ('{G.nodes[rep_node]['text']}'); "
                           f"guide: check every shift -> {'OVERDUE' if overdue else 'ok'} [{rep_node}]", 2 if overdue else 0)
        if cause == "nozzle_worn":
            h = job["nozzle_hours"]
            add(cause, f"nozzle operating hours {h} h; guide: replace every 40-60 h -> {'WORN (above 60 h)' if h > 60 else 'ok'} [{jn}]",
                3 if h > 100 else 2 if h > 60 else 0)
        # (e) settings against the cutting table and release rules
        if cause == "feed_too_high":
            too_high = job["is_fast"] or job["feed_vs_table"] > 1.10
            add(cause, f"program {job['program']}, feed {job['feed_vs_table']}x the cutting table; guide: max 1.10x, _FAST programs not released "
                       f"-> {'TOO HIGH' if too_high else 'ok'} [{jn}]", 4 if too_high else 0)
        if cause not in evidence:
            evidence[cause] = ["no data in the logs to check this cause"]

    return {"job_id": job_id, "defect": defect, "job": job, "candidates": cands, "evidence": evidence, "scores": scores}

r = retrieve("JOB-00391")
print(r["job_id"], "| defect:", r["defect"], "| candidates:", r["candidates"])
for c in r["candidates"]:
    print(f"\n{c}  (score {r['scores'][c]})")
    for line in r["evidence"][c]:
        print("   ", line)

JOB-00391 | defect: incomplete_cut | candidates: ['glass_dirty', 'feed_too_high', 'focus_wrong']

glass_dirty  (score 8)
    E1: alarm 2105 (Protective glass contamination) 10x in the 24 h before the job, last one 10.5 h before start; normal is 3.89/day -> UNUSUAL [alarm:145]
    E2: alarm 4001 (Laser power reduced) 4x in the 24 h before the job, last one -0.1 h before start; normal is 1.19/day -> UNUSUAL [alarm:146]
    E3: last glass cleaning 136.7 h before the job ('glass ok, wiped'); guide: check every shift -> OVERDUE [maint:29]

feed_too_high  (score 0)
    E4: program S235JR_12mm_BEV, feed 1.01x the cutting table; guide: max 1.10x, _FAST programs not released -> ok [job:JOB-00391]

focus_wrong  (score 0)
    no data in the logs to check this cause


## 3. Mode `kg_rules`: the graph alone
Pick the candidate with the most points. On a tie, take the one the guide lists first. No LLM involved.

In [4]:
def diagnose_rules(r):
    best = max(r["candidates"], key=lambda c: (r["scores"][c], -r["candidates"].index(c)))
    used = [line.split(":")[0] for line in r["evidence"][best] if line.startswith("E")]
    return {"cause": best, "evidence_ids": used, "fix": G.nodes[f"cause:{best}"]["fix"],
            "explanation": f"highest evidence score ({r['scores'][best]}) among {r['candidates']}"}

diagnose_rules(r)

{'cause': 'glass_dirty',
 'evidence_ids': ['E1', 'E2', 'E3'],
 'fix': 'Clean or replace protective glass',
 'explanation': "highest evidence score (8) among ['glass_dirty', 'feed_too_high', 'focus_wrong']"}

## 4. The prompt for the LLM
The LLM gets:
- a **system message** with its role and the rules (use only the evidence, pick from the list, answer in JSON),
- the job, its defect, the candidate causes with their guide text and fix, and the **evidence lines** from the graph.

Asking for **JSON** with fixed fields makes the answer easy to check automatically. `temperature 0` makes it give the same answer each run.

In [5]:
SYSTEM = (
    "You are a troubleshooting assistant for a TruLaser 3000 laser cutting machine. "
    "You diagnose the root cause of a cut-quality defect. Use ONLY the evidence given. "
    "Choose exactly one cause id from the candidate list. Prefer causes with UNUSUAL, OVERDUE, WORN, TOO HIGH or "
    "NO recalibration evidence; when evidence is equally strong, prefer the cause listed first. "
    'Answer with JSON only: {"cause": "<cause id>", "evidence_ids": ["E1", ...], "fix": "<action>", "explanation": "<one or two sentences>"}'
)

def prompt_kg(r):
    j = r["job"]
    lines = [f"Job {r['job_id']}: program {j['program']}, {j['material']} {j['thickness']} mm, gas {j['gas']}, "
             f"started {str(j['start'])[:16]}, {j['shift']} shift.",
             f"Inspection result: {r['defect']}.", "", "Candidate causes (troubleshooting guide order, most likely first):"]
    for i, c in enumerate(r["candidates"], 1):
        node = G.nodes[f"cause:{c}"]
        lines.append(f"{i}. {c}: {node['text']}. Fix: {node['fix']}. Rule: {node['rule']}.")
        lines += [f"   - {e}" for e in r["evidence"][c]]
    lines.append("\nWhich cause is most likely? Answer in JSON.")
    return "\n".join(lines)

print(prompt_kg(r))

Job JOB-00391: program S235JR_12mm_BEV, S235JR 12.0 mm, gas O2, started 2026-09-30 06:52, early shift.
Inspection result: incomplete_cut.

Candidate causes (troubleshooting guide order, most likely first):
1. glass_dirty: Protective glass contaminated or damaged (beam power reduced). Fix: Clean or replace protective glass. Rule: check every shift (8 h).
   - E1: alarm 2105 (Protective glass contamination) 10x in the 24 h before the job, last one 10.5 h before start; normal is 3.89/day -> UNUSUAL [alarm:145]
   - E2: alarm 4001 (Laser power reduced) 4x in the 24 h before the job, last one -0.1 h before start; normal is 1.19/day -> UNUSUAL [alarm:146]
   - E3: last glass cleaning 136.7 h before the job ('glass ok, wiped'); guide: check every shift -> OVERDUE [maint:29]
2. feed_too_high: Feed rate too high for the thickness. Fix: Return to cutting-table feed rate. Rule: max +10 % above cutting table; _FAST programs not released.
   - E4: program S235JR_12mm_BEV, feed 1.01x the cutting tab

### The prompt without the graph (`llm_only`)
Same question and the same guide text, but only the **job record**. No alarms, no maintenance, no comparison with the rules.
This is what an LLM with a plain manual (document RAG) would see.

In [6]:
def prompt_only(r):
    j = r["job"]
    lines = [f"Job {r['job_id']}: program {j['program']}, {j['material']} {j['thickness']} mm, gas {j['gas']}, "
             f"feed {j['feed']} mm/min, set gas pressure {j['gas_pressure_set']} bar, bevel angle {j['bevel_angle']} deg, "
             f"nozzle operating hours {j['nozzle_hours']}, started {str(j['start'])[:16]}, {j['shift']} shift.",
             f"Inspection result: {r['defect']}.", "", "Candidate causes (troubleshooting guide order, most likely first):"]
    for i, c in enumerate(r["candidates"], 1):
        node = G.nodes[f"cause:{c}"]
        lines.append(f"{i}. {c}: {node['text']}. Fix: {node['fix']}. Rule: {node['rule']}.")
    lines.append("\nNo machine log or maintenance data is available. Which cause is most likely? Answer in JSON (evidence_ids may be empty).")
    return "\n".join(lines)

print(prompt_only(r))

Job JOB-00391: program S235JR_12mm_BEV, S235JR 12.0 mm, gas O2, feed 1520.0 mm/min, set gas pressure 0.6 bar, bevel angle 22.5 deg, nozzle operating hours 14.2, started 2026-09-30 06:52, early shift.
Inspection result: incomplete_cut.

Candidate causes (troubleshooting guide order, most likely first):
1. glass_dirty: Protective glass contaminated or damaged (beam power reduced). Fix: Clean or replace protective glass. Rule: check every shift (8 h).
2. feed_too_high: Feed rate too high for the thickness. Fix: Return to cutting-table feed rate. Rule: max +10 % above cutting table; _FAST programs not released.
3. focus_wrong: Focus position wrong. Fix: Run focus test. Rule: not logged.

No machine log or maintenance data is available. Which cause is most likely? Answer in JSON (evidence_ids may be empty).


## 5. Ask the LLM
`ask_llm` sends the prompt to Ollama and reads the JSON answer. If the LLM names a cause that isn't in the list, we record `invalid`, which counts as wrong.

In [7]:
def ask_llm(user_prompt, candidates):
    resp = requests.post(f"{OLLAMA_URL}/api/chat", timeout=600, json={
        "model": MODEL, "stream": False, "format": "json", "options": {"temperature": 0},
        "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_prompt}]})
    text = resp.json()["message"]["content"]
    try:
        ans = json.loads(text)
    except json.JSONDecodeError:
        return {"cause": "invalid", "evidence_ids": [], "fix": "", "explanation": text[:300]}
    cause = str(ans.get("cause", "")).strip().lower()
    match = [c for c in candidates if c == cause or c in cause]
    ans["cause"] = match[0] if match else "invalid"
    return ans

if OLLAMA_OK:
    t0 = time.time()
    print(json.dumps(ask_llm(prompt_kg(r), r["candidates"]), indent=2))
    print(f"({time.time() - t0:.1f} s)")

{
  "cause": "glass_dirty",
  "evidence_ids": [
    "E1",
    "E2",
    "E3"
  ],
  "fix": "Clean or replace protective glass.",
  "explanation": "The protective glass has been contaminated or damaged, indicated by frequent alarms and overdue cleaning."
}
(6.5 s)


## 6. Run all bad jobs in the answer key
For every job in `answer_key_jobs.csv` and each mode, diagnose and save the answer.
Results are saved after every job to `results_<mode>.csv`. If you stop and re-run, **finished jobs are skipped**, so you never lose work.

With a laptop and an 8B model, expect a few seconds per job per mode. Set `LIMIT = 10` for a quick test first.

In [8]:
LIMIT = None        # e.g. 10 for a quick test; None = all jobs
MODES = ["kg_rules"] + (["kg_llm", "llm_only"] if OLLAMA_OK else [])

def run(mode):
    path = OUT / f"results_{mode}.csv"
    done = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["job_id"])
    todo = [j for j in key["job_id"] if j not in set(done["job_id"])][:LIMIT]
    rows = done.to_dict("records")
    for i, job_id in enumerate(todo, 1):
        r = retrieve(job_id)
        t0 = time.time()
        if mode == "kg_rules":
            ans = diagnose_rules(r)
        elif mode == "kg_llm":
            ans = ask_llm(prompt_kg(r), r["candidates"])
        else:
            ans = ask_llm(prompt_only(r), r["candidates"])
        rows.append({"job_id": job_id, "mode": mode, "defect": r["defect"], "cause": ans.get("cause"),
                     "evidence_ids": " ".join(map(str, ans.get("evidence_ids", []))), "fix": ans.get("fix", ""),
                     "explanation": ans.get("explanation", ""), "seconds": round(time.time() - t0, 2)})
        pd.DataFrame(rows).to_csv(path, index=False)
        if mode != "kg_rules" and i % 10 == 0:
            print(f"  {mode}: {i}/{len(todo)} done")
    print(f"{mode}: {len(rows)} jobs saved to {path.name}")

for mode in MODES:
    run(mode)

kg_rules: 99 jobs saved to results_kg_rules.csv
kg_llm: 99 jobs saved to results_kg_llm.csv


  llm_only: 10/88 done


  llm_only: 20/88 done


  llm_only: 30/88 done


  llm_only: 40/88 done


  llm_only: 50/88 done


  llm_only: 60/88 done


  llm_only: 70/88 done


  llm_only: 80/88 done


llm_only: 99 jobs saved to results_llm_only.csv


## 7. Read a few answers
Pick a job and compare what the three modes said with your answer sheet.

In [9]:
pd.set_option("display.max_colwidth", 160)
JOB = "JOB-00391"
frames = [pd.read_csv(OUT / f"results_{m}.csv") for m in MODES]
pd.concat(frames)[lambda d: d["job_id"] == JOB][["mode", "cause", "evidence_ids", "fix", "explanation"]]

,mode,cause,evidence_ids,fix,explanation
66,kg_rules,glass_dirty,E1 E2 E3,Clean or replace protective glass,"highest evidence score (8) among ['glass_dirty', 'feed_too_high', 'focus_wrong']"
66,kg_llm,glass_dirty,E1 E2 E3,Clean or replace protective glass.,"The protective glass has been contaminated or damaged, indicated by repeated alarms and overdue cleaning."
66,llm_only,feed_too_high,feed 1520.0 mm/min,Return to cutting-table feed rate,"The feed rate is 10% above the maximum allowed for this thickness, indicating a possible cause for the incomplete cut."


## What this notebook showed
- **Retrieval** is a graph walk: job → alarms before it → which causes they indicate; job → last repair → overdue or not; job → settings → against the guide's limits.
- The LLM receives that evidence as text with ids, so its answer can be **checked**: which evidence did it use?
- Running the same jobs **with and without the graph** is the experiment that shows whether the graph helps.

**Next (notebook 06):** score all three modes against your answer sheet.